# B0.2 — Prise en main de RDKit

**Objectif** (plan B, tâche B0.2) : savoir passer d'un SMILES à tout ce dont l'équipe a besoin, et vérifier la cohérence des identifiants du jeu de données.

1. Les fonctions de base de RDKit sur un exemple.
2. Recalculer l'InChIKey depuis `normalized_smiles` et comparer ses 14 premiers caractères à la colonne `inchikey14`, sur **tous** les SMILES distincts du train (le plan demandait un échantillon : sur 12 cœurs, le jeu complet prend moins d'une minute).
3. Compter les `inchikey14` qui ont plusieurs SMILES, et comprendre pourquoi.
4. Afficher 3 molécules avec un de leurs spectres, pour illustrer la fiche chimie.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Draw, rdFingerprintGenerator, rdMolDescriptors

# Racine du dépôt : premier dossier parent qui contient pyproject.toml
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from casmi.catalog import N_JOBS, RAW_DIR, smiles_properties, smiles_to_inchikey14

TRAIN_PATH = RAW_DIR / "train.parquet"
FIGURES_DIR = ROOT / "docs" / "figures"
print("Données :", TRAIN_PATH)
print("Processus pour RDKit :", N_JOBS)

## 1. Les fonctions de base, sur la caféine

In [ ]:
smiles = "Cn1cnc2c1c(=O)n(C)c(=O)n2C"
mol = Chem.MolFromSmiles(smiles)  # renvoie None si le SMILES est invalide

print("SMILES canonique  :", Chem.MolToSmiles(mol))
print("Formule brute     :", rdMolDescriptors.CalcMolFormula(mol))
print(f"Masse exacte      : {Descriptors.ExactMolWt(mol):.6f} Da (monoisotopique)")
print(f"Masse moyenne     : {Descriptors.MolWt(mol):.4f} Da (moyenne isotopique)")
print("InChIKey          :", Chem.MolToInchiKey(mol))
print("SMILES invalide   :", Chem.MolFromSmiles("C1CC"))

generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fp = generator.GetFingerprint(mol)
print(f"Fingerprint Morgan : {fp.GetNumOnBits()} bits à 1 sur {fp.GetNumBits()}")

theobromine = Chem.MolFromSmiles("Cn1cnc2c1c(=O)[nH]c(=O)n2C")
print(f"Tanimoto caféine / théobromine : {DataStructs.TanimotoSimilarity(fp, generator.GetFingerprint(theobromine)):.2f}")

Draw.MolToImage(mol, size=(300, 220))

### Pourquoi deux stéréoisomères ont le même `inchikey14`

L'InChIKey compte trois blocs : 14 caractères pour la connectivité (le squelette), puis 8 caractères pour le reste (stéréochimie, isotopes…), plus des indicateurs. La L-alanine et la D-alanine, images l'une de l'autre dans un miroir, n'ont donc que le deuxième bloc de différent.

In [ ]:
for name, smi in [("L-alanine", "C[C@H](N)C(=O)O"), ("D-alanine", "C[C@@H](N)C(=O)O"), ("alanine sans stéréo", "CC(N)C(=O)O")]:
    print(f"{name:20s} {smi:18s} {Chem.MolToInchiKey(Chem.MolFromSmiles(smi))}")

## 2. L'InChIKey recalculée correspond-elle à la colonne `inchikey14` ?

On recalcule l'InChIKey de chaque SMILES distinct du train, puis on compare ses 14 premiers caractères à la colonne `inchikey14`, spectre par spectre.

In [ ]:
pairs = pq.read_table(TRAIN_PATH, columns=["inchikey14", "normalized_smiles", "ingest_lib"]).to_pandas()
unique_smiles = pairs["normalized_smiles"].unique()

t0 = time.perf_counter()
recomputed = smiles_to_inchikey14(unique_smiles, n_jobs=N_JOBS)
print(f"{len(unique_smiles):,} SMILES distincts traités en {time.perf_counter() - t0:.0f} s sur {N_JOBS} processus")

pairs["inchikey14_rdkit"] = pairs["normalized_smiles"].map(recomputed)
match = pairs["inchikey14_rdkit"] == pairs["inchikey14"]
print(f"Spectres dont l'inchikey14 recalculé est identique : {match.mean():.4%} ({(~match).sum():,} différents)")

In [ ]:
mismatch = pairs.loc[~match, ["ingest_lib", "inchikey14", "inchikey14_rdkit", "normalized_smiles"]].drop_duplicates("normalized_smiles")
print(f"{len(mismatch):,} SMILES distincts en désaccord, par bibliothèque :")
print(pairs.loc[~match, "ingest_lib"].value_counts().to_string())
mismatch.head(10)

## 3. Les `inchikey14` qui ont plusieurs SMILES

On s'attendait surtout à des stéréoisomères. On vérifie d'abord si les SMILES du jeu contiennent de la stéréochimie (`@`, `/`, `\`), puis on classe les molécules qui ont plusieurs SMILES :
- **même molécule, écrite autrement** : RDKit donne le même SMILES canonique ;
- **formules différentes** : forme protonée ou chargée, ou autre état ;
- **tautomères** : même formule, mais un hydrogène placé ailleurs (un H « mobile », que la couche standard de l'InChI ne distingue pas).

In [ ]:
distinct = pairs[["inchikey14", "normalized_smiles"]].drop_duplicates()
has_stereo = distinct["normalized_smiles"].str.contains(r"[@/\\]", regex=True)
print(f"SMILES distincts avec de la stéréochimie : {has_stereo.sum():,} sur {len(distinct):,}")

n_smiles = distinct.groupby("inchikey14").size()
multi = distinct[distinct["inchikey14"].isin(n_smiles.index[n_smiles > 1])].copy()
print(f"Molécules avec plusieurs SMILES : {(n_smiles > 1).sum():,} sur {len(n_smiles):,} ({(n_smiles > 1).mean():.2%})")

multi["canonique"] = multi["normalized_smiles"].map(lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)))
multi["formula"] = multi["normalized_smiles"].map(smiles_properties(multi["normalized_smiles"].unique(), n_jobs=1)["formula"])

per_key = multi.groupby("inchikey14").agg(n_canoniques=("canonique", "nunique"), n_formules=("formula", "nunique"))
per_key["catégorie"] = np.select(
    [per_key["n_canoniques"] == 1, per_key["n_formules"] > 1],
    ["même molécule, écrite autrement", "formules différentes (protonation, charge)"],
    default="tautomères (même formule)",
)
per_key["catégorie"].value_counts().to_frame("n_molécules")

In [ ]:
example_key = per_key.index[per_key["catégorie"] == "tautomères (même formule)"][0]
example = multi.loc[multi["inchikey14"] == example_key, "normalized_smiles"].tolist()
print("Exemple de tautomères partageant le même inchikey14 :", example_key)
for s in example:
    print("  ", s, "->", Chem.MolToInchiKey(Chem.MolFromSmiles(s)))
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in example], molsPerRow=len(example), subImgSize=(260, 200))

## 4. Trois molécules et un de leurs spectres

Pour la fiche chimie : la structure, le précurseur, l'adduct et les fragments. On choisit trois molécules connues présentes dans le train.

In [ ]:
EXAMPLES = {
    "Caféine": "RYYVLZVUVIJVGH",
    "Tryptophane": "QIVBCDIJIAJPQS",
    "Quercétine": "REFJWTPEDVJJIY",
}

spectra = pq.read_table(
    TRAIN_PATH,
    columns=["inchikey14", "normalized_smiles", "ingest_lib", "adduct", "precursor_mz", "collision_energy_ev",
             "ms2_mzs", "ms2_normalized_intensities"],
    filters=[("inchikey14", "in", list(EXAMPLES.values())), ("adduct", "=", "[M+H]+")],
).to_pandas()
print(spectra.groupby("inchikey14").size().rename("spectres [M+H]+ dans le train").to_string())

Attention : `collision_energy_ev` n'est pas un nombre mais une **liste** par spectre. Quand l'acquisition est « étagée », le précurseur est fragmenté successivement à plusieurs énergies et les fragments sont cumulés dans un seul spectre. C'est à signaler à A pour les features de métadonnées.

In [ ]:
energies = pq.read_table(TRAIN_PATH, columns=["collision_energy_ev"]).column(0).to_pandas()
energies.map(lambda e: 0 if e is None else len(e)).value_counts().sort_index().rename_axis("nombre d'énergies").to_frame("spectres")

In [ ]:
fig, axes = plt.subplots(len(EXAMPLES), 2, figsize=(11, 3.2 * len(EXAMPLES)),
                         gridspec_kw={"width_ratios": [1, 2.4]})

for (name, key), (ax_mol, ax_spec) in zip(EXAMPLES.items(), axes):
    # Un spectre « propre » : aucun pic au-dessus du précurseur (un fragment est forcément plus
    # léger que l'ion de départ), une seule énergie entre 15 et 45 eV, et un nombre de pics médian
    rows = spectra[spectra["inchikey14"] == key]
    clean = rows[
        (rows["ms2_mzs"].map(max) <= rows["precursor_mz"] + 0.5)
        & rows["collision_energy_ev"].map(lambda e: e is not None and len(e) == 1 and 15 <= e[0] <= 45)
    ]
    clean = clean.assign(n=clean["ms2_mzs"].map(len)).sort_values("n")
    row = clean.iloc[len(clean) // 2]
    mol = Chem.MolFromSmiles(row["normalized_smiles"])

    ax_mol.imshow(Draw.MolToImage(mol, size=(320, 240)))
    ax_mol.axis("off")
    ax_mol.set_title(f"{name}\n{rdMolDescriptors.CalcMolFormula(mol)}, M = {Descriptors.ExactMolWt(mol):.4f} Da",
                     fontsize=10, loc="left")

    mzs, intensities = np.asarray(row["ms2_mzs"]), np.asarray(row["ms2_normalized_intensities"])
    ax_spec.vlines(mzs, 0, intensities, color="#2a78d6", linewidth=1.2)
    ax_spec.axvline(row["precursor_mz"], color="#52514e", linestyle="--", linewidth=1)
    ax_spec.text(row["precursor_mz"], 1.02, f"précurseur {row['precursor_mz']:.4f}", ha="right",
                 color="#52514e", fontsize=8, transform=ax_spec.get_xaxis_transform())
    for i in np.argsort(intensities)[-3:]:
        ax_spec.annotate(f"{mzs[i]:.3f}", (mzs[i], intensities[i]), xytext=(0, 3),
                         textcoords="offset points", ha="center", fontsize=8, color="#52514e")
    # collision_energy_ev est une liste : plusieurs énergies quand l'acquisition est « étagée »
    energies = row["collision_energy_ev"]
    energy = f", {' / '.join(f'{e:.0f}' for e in energies)} eV" if energies is not None and len(energies) else ""
    ax_spec.set_title(f"Spectre MS/MS, {row['adduct']}{energy} ({row['ingest_lib']})", fontsize=10, loc="left")
    ax_spec.set_xlim(0, row["precursor_mz"] * 1.08)
    ax_spec.set_ylim(0, 1.15)
    ax_spec.set_ylabel("intensité relative")
    ax_spec.spines[["top", "right"]].set_visible(False)

axes[-1][1].set_xlabel("m/z")
fig.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "exemples_spectres.png", dpi=120, bbox_inches="tight")
plt.show()

## Conclusions

1. **Les identifiants du jeu sont cohérents.** L'`inchikey14` recalculé par RDKit depuis `normalized_smiles` est identique à la colonne `inchikey14` pour **100 % des spectres** (277 566 SMILES distincts, 22 s sur 12 cœurs).
2. **Le jeu de données est en 2D.** Aucun des 277 566 SMILES ne contient de stéréochimie. Une « molécule » au sens de la compétition est donc une structure 2D, et les stéréoisomères (comme la L- et la D-alanine) sont confondus, ce qui est cohérent avec la tâche : identifier la structure 2D.
3. **1 612 molécules (0,58 %) ont plusieurs SMILES**, et ce ne sont jamais deux écritures du même graphe :
   - 1 494 sont des **tautomères**, avec la même formule et un hydrogène déplacé, par exemple `C(O)=N` contre `C(=O)N`. L'InChI standard ne les distingue pas, d'où le même `inchikey14`, mais leurs fingerprints de Morgan peuvent différer ;
   - 118 ont des **formules différentes**, à un proton près : forme protonée et forme neutre de la même molécule.
4. **`collision_energy_ev` est une liste** : 67 % des spectres ont une seule énergie, 19,5 % plusieurs (acquisition « étagée »), et 13 % aucune. **À signaler à A** pour `meta_features`.
5. La figure des trois exemples est enregistrée dans `docs/figures/exemples_spectres.png`, pour la fiche chimie.